# Lab 11B: An eval and debugging harness
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 11 · about 20 minutes**

**Objectives**
- Build a labelled eval set with code graders
- Add a rubric-based LLM grader
- Compare two prompt versions per example
- Trigger, diagnose, and log three common failures

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Setup: run once per session
!pip -q install -U anthropic 
import os, json, time, random, importlib, re
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in your environment"
import anthropic
print("anthropic SDK", anthropic.__version__, "| key loaded:", bool(os.environ.get("ANTHROPIC_API_KEY")))

In [ ]:
%%writefile config.py
"""One place for Claude settings (same pattern as Day 1)."""
import os
# Confirm current IDs on the models overview page: https://platform.claude.com/docs/en/about-claude/models/overview
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5")
FAST_MODEL = os.environ.get("CLAUDE_FAST_MODEL", "claude-haiku-4-5-20251001")
MAX_TOKENS = 1024
MAX_RETRIES = 2
TIMEOUT_S = 120

In [ ]:
import config
importlib.reload(config)
client = anthropic.Anthropic(max_retries=config.MAX_RETRIES, timeout=config.TIMEOUT_S)

def log_response(resp, label=""):
    kinds = [b.type for b in resp.content]
    print(f"{label} stop_reason={resp.stop_reason} blocks={kinds} "
          f"input_tokens={resp.usage.input_tokens} output_tokens={resp.usage.output_tokens}")
    for b in resp.content:
        if b.type == "text":
            print("   ", b.text[:300].replace("\n", " "))

def text_of(resp):
    return "".join(b.text for b in resp.content if b.type == "text")

print("model:", config.MODEL, "| fast model:", config.FAST_MODEL)

## Drill 1: an eval set with code graders

In [ ]:
EVAL = [
 ("Hi, order ORD-1182 charged me 49.99 twice, please refund one.", {"order_id": "ORD-1182", "amount": 49.99, "intent": "refund"}),
 ("Where is my router? Order ORD-2201.", {"order_id": "ORD-2201", "amount": None, "intent": "status"}),
 ("Cancel ORD-3307 please, the 120 USD plan is too expensive.", {"order_id": "ORD-3307", "amount": 120.0, "intent": "cancel"}),
 ("ord-4410 arrived broken, I paid $89.50, want my money back", {"order_id": "ORD-4410", "amount": 89.5, "intent": "refund"}),
 ("Can you tell me when ORD-5521 ships?", {"order_id": "ORD-5521", "amount": None, "intent": "status"}),
 ("I want to stop order ORD-6603 before it ships.", {"order_id": "ORD-6603", "amount": None, "intent": "cancel"}),
 ("Refund 15 dollars on ORD-7714, wrong add-on charged.", {"order_id": "ORD-7714", "amount": 15.0, "intent": "refund"}),
 ("Status update for ORD 8820?", {"order_id": "ORD-8820", "amount": None, "intent": "status"}),
]
SCHEMA = {"type": "object", "properties": {"order_id": {"type": "string"}, "amount": {"type": ["number", "null"]},
          "intent": {"type": "string", "enum": ["refund", "status", "cancel", "other"]}}, "required": ["order_id", "amount", "intent"], "additionalProperties": False}
PROMPTS = {
 "v1": "Extract the order ID, amount, and intent from the message as JSON.",
 "v2": "Extract fields from a telecom order message. order_id: normalize to ORD-1234 format. amount: a number in USD, or null if no amount is stated. "
       "intent: refund, status, cancel, or other. Never invent an amount.",
}
def extract(version, text):
    r = client.messages.create(model=config.FAST_MODEL, max_tokens=200, system=PROMPTS[version],
                               output_config={"format": {"type": "json_schema", "schema": SCHEMA}}, messages=[{"role": "user", "content": text}])
    return json.loads(text_of(r))
def grade(pred, gold):
    amt_ok = (pred["amount"] is None and gold["amount"] is None) or (pred["amount"] is not None and gold["amount"] is not None and abs(pred["amount"] - gold["amount"]) < 0.01)
    return {"order_id": pred["order_id"] == gold["order_id"], "amount": amt_ok, "intent": pred["intent"] == gold["intent"]}
RES = {}
for v in PROMPTS:
    RES[v] = [grade(extract(v, t), g) for t, g in EVAL]
    print(v, {f: f"{sum(r[f] for r in RES[v])}/{len(EVAL)}" for f in ["order_id", "amount", "intent"]})

## Drill 2: a rubric-based LLM grader

In [ ]:
SOURCE = "Outage report: sector 9 lost fiber from 05:10 to 09:40 after a cable cut. 1,240 customers affected. Credits of one day will be applied automatically."
SUMMARIES = {"good": "Sector 9 lost fiber from 05:10 to 09:40 due to a cable cut, affecting 1,240 customers; a one-day credit is applied automatically.",
             "missing facts": "There was an outage in sector 9 this morning.",
             "invented": "Sector 9 lost fiber for two days, affecting 5,000 customers, who must call to claim a refund."}
RUBRIC = {"type": "object", "properties": {"completeness": {"type": "integer"}, "accuracy": {"type": "integer"}, "unsupported_claims": {"type": "array", "items": {"type": "string"}}},
          "required": ["completeness", "accuracy", "unsupported_claims"], "additionalProperties": False}
GRADER = ("Score 1 to 5. completeness: 5 = time window, cause, customers affected, and credit all present; 1 = most missing. "
          "accuracy: 5 = every claim matches the source; 1 = several contradictions. List unsupported claims.")
grades = {k: json.loads(text_of(client.messages.create(model=config.MODEL, max_tokens=300, system=GRADER,
          output_config={"format": {"type": "json_schema", "schema": RUBRIC}}, messages=[{"role": "user", "content": f"SOURCE: {SOURCE}\nSUMMARY: {s}"}]))) for k, s in SUMMARIES.items()}
pd.DataFrame(grades).T

## Drill 3: per-example regression
The average can improve while individual cases get worse. Look at every row.

In [ ]:
rows = []
for i, (t, g) in enumerate(EVAL):
    a, b = all(RES["v1"][i].values()), all(RES["v2"][i].values())
    rows.append({"example": t[:48], "v1 all fields": a, "v2 all fields": b, "change": "fixed" if b and not a else "REGRESSED" if a and not b else "same"})
reg = pd.DataFrame(rows)
print("v1 fully correct:", int(reg["v1 all fields"].sum()), "| v2 fully correct:", int(reg["v2 all fields"].sum()), "| regressions:", int((reg.change == "REGRESSED").sum()))
reg

## Drill 4: trigger, diagnose, and log three failures

In [ ]:
def diagnose(err):
    msg = str(err).lower()
    if "tool_use" in msg and "tool_result" in msg: return "A tool call was not answered: return a tool_result for every tool_use id."
    if getattr(err, "status_code", None) == 404 or "not_found" in msg: return "Model ID is retired or mistyped: check the models page and config.py."
    if getattr(err, "status_code", None) == 400: return "Request error: fix the request; do not retry it unchanged."
    return "Unknown: check the request ID in logs."
def logged(fn, label):
    try:
        r = fn(); rec = {"case": label, "request_id": getattr(r, "_request_id", None), "stop_reason": r.stop_reason, "diagnosis":
                         "Output truncated: raise max_tokens or ask for less." if r.stop_reason == "max_tokens" else "ok"}
    except anthropic.APIStatusError as e:
        rec = {"case": label, "request_id": getattr(e, "request_id", None), "status": e.status_code, "diagnosis": diagnose(e)}
    return rec
TOOLS = [{"name": "lookup_order", "description": "Look up an order by ID.", "input_schema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}]
def missing_tool_result():
    first = client.messages.create(model=config.MODEL, max_tokens=300, tools=TOOLS, tool_choice={"type": "any"}, messages=[{"role": "user", "content": "Where is ORD-1182?"}])
    return client.messages.create(model=config.MODEL, max_tokens=300, tools=TOOLS, messages=[{"role": "user", "content": "Where is ORD-1182?"},
                                  {"role": "assistant", "content": first.content}, {"role": "user", "content": "Any update?"}])
CASES = [(missing_tool_result, "missing tool_result"),
         (lambda: client.messages.create(model="claude-retired-model-x", max_tokens=50, messages=[{"role": "user", "content": "hi"}]), "bad model ID"),
         (lambda: client.messages.create(model=config.FAST_MODEL, max_tokens=12, messages=[{"role": "user", "content": "Explain fiber networks in detail."}]), "truncation")]
pd.DataFrame([logged(fn, label) for fn, label in CASES])

Each diagnosis should become an eval or test case, so the same failure cannot silently return.

## You should now have
- [ ] Field-level accuracy from code graders
- [ ] Rubric grades with unsupported claims listed
- [ ] A per-example regression table
- [ ] Three diagnosed failures with request IDs

## Check yourself
1. Which grader caught the invented summary, and how?
2. Did v2 regress anywhere? What would you do about it?
3. Which of the three failures have you seen in real projects?